In [1]:
nq = 3
ns = 7
seed = 3
case_id = f"q{nq}_n{ns}_s{seed}"
qasm_name = f"qc_iso_{case_id}_no_backend.qasm"

In [2]:
from qiskit.circuit import QuantumCircuit

qc = QuantumCircuit.from_qasm_file(qasm_name)
print(qc.count_ops())
print(qc.depth())

OrderedDict({'u': 122, 'cx': 73})
145


In [3]:
# https://docs.quantum.ibm.com/guides/synthesize-unitary-operators#synthesize-unitary-operations
from qiskit.quantum_info import Operator
 
# compute unitary matrix of circuit
U = Operator(qc)

# re-synthesize
resynth_circuit = QuantumCircuit(nq + 1)
resynth_circuit.unitary(U, range(nq + 1))
# better_circuit.decompose().draw()


In [4]:
resynth_circuit = resynth_circuit.decompose(reps=3)
print(resynth_circuit.count_ops())
print(resynth_circuit.depth())

OrderedDict({'u': 208, 'cx': 100})
225


In [5]:
U = Operator(qc)

tmp_circuit = QuantumCircuit(nq + 1)
tmp_circuit.unitary(U, range(nq + 1))

from qiskit import transpile

approx_circuit = transpile(
    tmp_circuit,
    unitary_synthesis_method="aqc",
    approximation_degree=0,
)

approx_circuit = approx_circuit.decompose(reps=3)
# better_circuit.decompose().draw()
print(approx_circuit.count_ops())
print(approx_circuit.depth())

OrderedDict({'u': 126, 'cx': 61})
83


In [6]:
# Approx again
U = Operator(approx_circuit)

better_circuit_2 = QuantumCircuit(nq + 1)
better_circuit_2.unitary(U, range(nq + 1))


approx_twice_circuit = transpile(
    better_circuit_2,
    unitary_synthesis_method="aqc",
    unitary_synthesis_plugin_config={
        # "network_layout": "cart",
        "connectivity_type": "star",
        "depth": 10,
    },
    approximation_degree=0,
)

approx_twice_circuit = approx_twice_circuit.decompose(reps=3)
# better_circuit.decompose().draw()
print(approx_twice_circuit.count_ops())
print(approx_twice_circuit.depth())
# approx_twice_circuit.draw()

OrderedDict({'u': 24, 'cx': 10})
15


## Fidelity

In [ ]:
from qiskit.quantum_info import process_fidelity

# Two operators which differ only by phase
op_a = Operator(qc)
op_b = Operator(resynth_circuit)
op_c = Operator(approx_circuit)
op_d = Operator(approx_twice_circuit)
 
# Compute process fidelity
F_resynth = process_fidelity(op_a, op_b)
print("Process fidelity (resynth) =", F_resynth)
F_approx = process_fidelity(op_a, op_c)
print("Process fidelity (approx) =", F_approx)
F_approx_twice = process_fidelity(op_a, op_d)
print("Process fidelity (approx_twice) =", F_approx_twice)

Process fidelity (resynth) = 0.9999999999999978
Process fidelity (approx) = 0.9999841424128181
Process fidelity (approx_twice) = 0.4192991260650874


In [8]:
import qiskit.qasm2
qiskit.qasm2.dump(
    resynth_circuit,
    f"qc_iso_{case_id}_no_backend_resynth.qasm",
)
qiskit.qasm2.dump(
    approx_circuit,
    f"qc_iso_{case_id}_no_backend_approx.qasm",
)

## Summary

In [10]:
print(qc.count_ops())
print(qc.depth())
print(resynth_circuit.count_ops())
print(resynth_circuit.depth())
print(approx_circuit.count_ops())
print(approx_circuit.depth())
print(approx_twice_circuit.count_ops())
print(approx_twice_circuit.depth())

OrderedDict({'u': 122, 'cx': 73})
145
OrderedDict({'u': 208, 'cx': 100})
225
OrderedDict({'u': 126, 'cx': 61})
83
OrderedDict({'u': 24, 'cx': 10})
15
